# Can we predict late delivery?
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), 5-fold walk-forward CV on training rows, selection by mean CV ROC AUC.

Excluded: `consignment` (identifier), `dispatched_on` (split only), and **`claim_paid_eur`** — compensation is settled after delivery, so it is not known at prediction time (leakage).

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].values
assert len(df)==8051 and len(hold_idx)==1610
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
print(len(train_idx),len(hold_idx), df.late_delivery.mean())
print('train max date',df.loc[train_idx,'dispatched_on'].max(),'holdout min date',df.loc[hold_idx,'dispatched_on'].min())
print(df.isna().sum())

In [ ]:
TARGET='late_delivery'
NUM=['distance_km','weight_kg','parcels']; CAT=['carrier']
FEATURES=NUM+CAT
# training rows in time order (walk-forward CV needs chronological order)
tr=df.loc[train_idx].sort_values('dispatched_on',kind='stable')
X,y=tr[FEATURES],tr[TARGET]; tr_labels=tr.index.values
pre_lin=ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
pre_tree=ColumnTransformer([('n','passthrough',NUM),('c',OneHotEncoder(handle_unknown='ignore'),CAT)])
# all candidates named before any is scored
candidates={
 'baseline_prior':make_pipeline(pre_tree,DummyClassifier(strategy='prior')),
 'logreg_C1':make_pipeline(pre_lin,LogisticRegression(C=1.0,max_iter=1000)),
 'logreg_C0.1':make_pipeline(pre_lin,LogisticRegression(C=0.1,max_iter=1000)),
 'random_forest':make_pipeline(pre_tree,RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1)),
 'hist_gb':make_pipeline(pre_tree,HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)),
}
list(candidates)

In [ ]:
# walk-forward CV inside training rows; preprocessing is part of each pipeline, so fitted per fold
from sklearn.model_selection import TimeSeriesSplit
tscv=TimeSeriesSplit(n_splits=5)
cv_folds=[]; cv_scores={}
for name,m in candidates.items():
    s=[]
    for k,(a,b) in enumerate(tscv.split(X)):
        mm=clone(m).fit(X.iloc[a],y.iloc[a])
        s.append(float(roc_auc_score(y.iloc[b],mm.predict_proba(X.iloc[b])[:,1])))
        if name=='baseline_prior':
            cv_folds.append([[int(i) for i in tr_labels[a]],[int(i) for i in tr_labels[b]]])
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv

In [ ]:
chosen=cv['mean'].idxmax(); print('chosen:',chosen)
final=clone(candidates[chosen]).fit(X,y)
ho=df.loc[hold_idx]
auc=float(roc_auc_score(ho[TARGET],final.predict_proba(ho[FEATURES])[:,1]))  # scored once
print('holdout roc_auc',auc)

In [ ]:
evidence={'features':FEATURES,'candidates':list(candidates),'cv_folds':cv_folds,
 'cv_scores':cv_scores,'chosen':chosen,'holdout_rows':[int(i) for i in hold_idx],
 'holdout_scores':{chosen:auc}}
json.dump(evidence,open('evidence.json','w'))